# Spaghetti
Make spaghetti plots for ENSO events

## Imports

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import copy
import pandas as pd

# Import custom modules
import src.utils

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Funcs

In [ ]:
def get_spaghetti(idx, data, peak_month, event_type=None, q=0.95, is_warm=True):
    """
    Get hovmoller composite based on specified:
    - data: used to compute index/make composite
    - peak_month: month to center composite on
    - q: quantile threshold for composite
    """

    ## handle warm/cold case
    if is_warm:
        kwargs = dict(q=q, check_cutoff=lambda x, cut: x > cut)
    else:
        kwargs = dict(q=1 - q, check_cutoff=lambda x, cut: x < cut)

    ## kwargs for composite
    kwargs = dict(
        kwargs,
        avg=False,
        peak_month=peak_month,
        idx=idx,
        data=data,
        event_type=event_type,
    )

    ## composite of projected data
    spag = src.utils.make_composite(**kwargs)

    return spag


def save(fig, fname, dpi=300, do_save = False):
    """save figure to file"""

    ## get save directory
    save_dir = pathlib.Path(os.environ["SAVE_FP"], "ch3-outline")

    ## get fname
    fname = save_dir / f"{fname}.pdf"

    if do_save:
        fig.savefig(fname, dpi=dpi)
        
    return

def load_consolidated_wide():
    """utility function to load consolidated data"""

    ## directory with data
    CONS_DIR = pathlib.Path(os.environ["DATA_FP"], "cesm", "consolidated_05")

    ## function to align and open
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    align_and_open = lambda fp: src.utils.align_pop_times(xr.open_dataset(fp), **kwargs)

    ## open data and align pop times
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    forced = align_and_open(CONS_DIR / "forced.nc")
    anom = align_and_open(CONS_DIR / "anom.nc")

    return forced, anom

def load_h_ohc_idxs():
    """load OHC-based h indices"""

    ## specify subsetting funcs
    LONS_E = dict(longitude=slice(210, 270))
    LONS_W = dict(longitude=slice(120, 180))

    ## averaging funcs
    lon_avg = lambda x, lons: x.sel(lons).mean("longitude")
    lat_avg = lambda x: x.sel(latitude=slice(-5, 5)).mean("latitude")
    
    ## load spatial data
    _, anom_wide = load_consolidated_wide()

    ## empty array to hold result
    h_idxs = xr.Dataset()

    ## compute ohc
    h_idxs["h_w_ohc"] = src.utils.reconstruct_wrapper(
        anom_wide[["T", "T_comp"]],
        lambda x: lon_avg(x.integrate("z_t"), LONS_W) / 300,
    )["T"]
    h_idxs["h_e_ohc"] = src.utils.reconstruct_wrapper(
        anom_wide[["T", "T_comp"]],
        lambda x: lon_avg(x.integrate("z_t"), LONS_E) / 300,
    )["T"]

    h_idxs["h_ohc"] = .5*(h_idxs["h_w_ohc"]+h_idxs["h_e_ohc"])
    h_idxs["dh_ohc"] = h_idxs["h_e_ohc"]-h_idxs["h_w_ohc"]

    return h_idxs

## Load data

### T, h

In [ ]:
## fits

## Th data
Th = src.utils.load_cesm_indices(load_z20=True)

## load ohc data
Th = xr.merge([Th, load_h_ohc_idxs()])

## omit first year (bc of NaN in h,hw vars)
Th = Th.sel(time=slice("1851", None))

#### Load heat flux

In [ ]:
_, anom = src.utils.load_consolidated()

In [ ]:
## get NHF in Niño 3.4 region
nhf_34 = src.utils.reconstruct_wrapper(
    anom[["nhf","nhf_comp"]], fn=src.utils.get_nino34
)["nhf"]

## convert to units of K/mo
sec_per_mo = 8.64e4 * 30
sec_per_yr = 12 * sec_per_mo
rho = 1.02e3
Cp = 4.2e3
H = 50

Q = nhf_34 * sec_per_yr / (rho * Cp * H)

## merge with Th data
Th["Q"] = Q.sel(time=Th.time)

#### get windowed data

In [ ]:
## get windowed data (used to estimate change in parameters over time)
Th_rolling = src.utils.get_windowed(Th, window_size=480, stride=120)

### Early and late

In [ ]:
## specify years
years = [2011, 2081]

## get early and late
Th_early = Th_rolling.sel(year=years[0])
Th_late = Th_rolling.sel(year=years[1])

## remove median
Th_early = Th_early.groupby("time.month")-Th_early.groupby("time.month").median()
Th_late = Th_late.groupby("time.month")-Th_late.groupby("time.month").median()

## compute spaghetti/composite

In [ ]:
## specify composite specs
comp_kwargs = dict(
    is_warm=True,
    event_type=None,
    peak_month=12,
    q=0.95,
)

## specify variable to use for composite
VARNAME = "T_34"

## do the compute
spag_early = get_spaghetti(idx=Th_early[VARNAME], data=Th_early, **comp_kwargs).drop_vars("year")
spag_late = get_spaghetti(idx=Th_late[VARNAME], data=Th_late, **comp_kwargs).drop_vars("year")

## plots

### Mean

Clean up; add bounds

In [ ]:
## get data
get = lambda x : x["T_34"]
spags = xr.merge([get(spag_early).rename("early"), get(spag_late).rename("late")])

## get quantiles
spags_q = spags.quantile(q=[.1,.5,.9],dim="sample").rename({"quantile":"q"})

# sel = lambda x : 12 * x.differentiate("lag")
# spags_q = sel(spags_q)

## specify colors
colors = sns.color_palette()
# colors = ["k",colors[1]]

fig, ax = plt.subplots(figsize=(2, 2.5), layout="constrained")

for color, n, l in zip(colors, ["early", "late"], ["early", "late"]):
    ax.plot(spags_q[n].sel(q=.5), spags_q.lag, label=l, color=color, lw=4)

y1, y2 = [spags_q.sel(q=q) for q in [.1, .9]]
ax.fill_betweenx(y=spags_q.lag, x1=y1["early"], x2=y2["early"], color=colors[0], alpha=.1)

for y_ in [y1, y2]:
    # ax.plot( y_["early"], y_.lag, lw=.8, color=colors[0], ls="-")
    ax.plot(y_["late"], y_.lag, lw=.8, color=colors[1], ls="-")

## fill between
# for y_ in [-2,0]:
#     ax.axhline
# ax.fill_between(x =np.linspace(*ax.get_xlim()), y1=-2, y2=0, color="k", alpha=.1)

    # for q in [.1, .9]:
    #     ax.plot(spag.lag, spag["T_34"].quantile(q=q, dim="sample"), ls = "--", color=color, lw=1)

## formatting
f_kwargs = dict(ls="--", c="k", lw=0.5)
for y_ in [-2,1]:
    ax.axhline(y_, **f_kwargs)
ax.axvline(0, **f_kwargs)
ax.set_ylabel("Lag (months)")
ax.axvline(-3, c="gray", lw=0.5)
ax.set_yticks([-6, -2, 1, 6])
ax.set_xticks([0,3])
ax.set_xlim([-.5,None])
ax.set_ylim([-6,6])
# ax.set_ylim([-.5, None])

plt.show()

In [ ]:
def prep(spag):
    """prep spaghetti"""

    ## differentiate / normalize
    spag["ddt_T"] = 12 * spag["T_34"].differentiate("lag")
    spag["Q_gr"] = spag["Q"] / spag["T_34"]
    spag["gr"] = spag["ddt_T"] / spag["T_34"]

    ## rolling
    return spag.rolling({"lag":3}, center=True).mean()
    # return spag

spag_diff = prep(spag_late)-prep(spag_early)
spag_diff_q = spag_diff.quantile(q=[.1,.5,.9],dim="sample").rename({"quantile":"q"})

In [ ]:
def relu_neg(x):
    """get (negative) relu of dataarray"""
    return x.where(x<0, other=0)

In [ ]:
## specify y-lim
xlim = [-7.2,3]
ylim = [-6,6]

fig, axs = plt.subplots(1,3, figsize=(4.5, 2), layout="constrained")

## plot difference
for i, (sp, c) in enumerate(zip([spag_early, spag_late], colors)):
    sp_ = prep(sp)["T_34"].quantile(q=[.1,.5,.9], dim="sample").rename({"quantile":"q"})
    # for q, lw in zip(sp_.q, [.7,4,.7]):
    axs[0].plot(sp_.sel(q=.5), sp.lag, lw=4, c=c)

    ## plot bounds
    lb, ub = [sp_.sel(q=q) for q in [.1, .9]]
    if i:
        for q in sp_.q:
            axs[0].plot(sp_.sel(q=q), sp.lag, lw=1, c=c)
    else:
        axs[0].fill_betweenx(y=sp_.lag, x1=lb, x2=ub, color=c, alpha=.1)

for ax, v, in zip(axs[1:], ["ddt_T","Q"]):
    for q, lw in zip([.5, .1, .9], [4,1,1]):
        ax.plot(spag_diff_q[v].sel(q=q), spags_q.lag, lw=lw, c=colors[4])

    ## highlight damping
    med = spag_diff_q[v].sel(q=.5)
    ax.fill_betweenx(
        y=med.lag, x2=relu_neg(med), x1=np.zeros_like(med.lag), color="k", alpha=.1, lw=0)
    ax.set_yticks([])
    ax.set_xlabel(r"K yr$^{-1}$")
    ax.set_xticks([-6,0])
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)



axs[0].set_xlabel(r"K")
axs[0].set_ylabel("Lag (months)")
axs[0].set_xticks([0, 3])
axs[0].set_yticks([-6, -2, 1, 6])
axs[0].set_ylim(ylim)
axs[0].set_xlim([-.5,None])
f_kwargs = dict(lw=.75, c="k", ls="--")
for ax in axs:
    ax.axvline(0, c="k", lw=1)
    ax.axhline(1, **f_kwargs)
    ax.axhline(-2, **f_kwargs)



plt.show()

In [ ]:
min(3,45)

In [ ]:
def get_pdf_diff(pdf):
    """Get difference between PDFs"""

    ## get difference between medians
    m = pdf["late"].sel(q=.5) - pdf["early"].sel(q=.5)

    ## get difference between combinations of bounds
    d0 = pdf["late"].sel(q=.1)-pdf["early"].sel(q=.9)
    d1 = pdf["late"].sel(q=.9)-pdf["early"].sel(q=.1)
    e = xr.concat([d0, d1], dim=pd.Index([0,1],name="diff"))
                  
    lb = e.min("diff")
    ub = e.max("diff")

    return m, lb, ub

In [ ]:
m,lb,ub = get_pdf_diff(spags_Q)

In [ ]:
fig,ax = plt.subplots(figsize=(3,2.5))
plt.plot(m.sel(lag=slice(-6,6)), c="k")
plt.plot(ub.sel(lag=slice(-6,6)))
plt.plot(lb.sel(lag=slice(-6,6)))
plt.axhline(0, lw=1, c="gray")
plt.show()

In [ ]:
edges = np.arange(-15,11,1.5)
pdf0,_ = src.utils.get_empirical_pdf(spags_Q_["early"].sel(lag=0).values, edges=edges)
pdf1,_ = src.utils.get_empirical_pdf(spags_Q_["late"].sel(lag=0).values, edges=edges)

# edges_diff = np.arange(-5,6,1)
spags_diff = (spags_Q_["late"]-spags_Q_["early"])
pdf_diff,_ = src.utils.get_empirical_pdf(spags_diff.sel(lag=0).values, edges=edges)
plt.show()

In [ ]:
fig,ax = plt.subplots(figsize=(4,3))
ax.stairs(pdf0, edges)
ax.stairs(pdf1, edges)
plt.show()
fig,ax = plt.subplots(figsize=(4,3))
ax.stairs(pdf_diff, edges)
# ax.stairs(pdf1, edges)
plt.show()

In [ ]:
fig,ax = plt.subplots(figsize=(3,2.5))
plt.plot(m.sel(lag=slice(-6,6)), c="k")
plt.plot(ub.sel(lag=slice(-6,6)))
plt.plot(lb.sel(lag=slice(-6,6)))

plt.plot(spags_diff.quantile(q=.5, dim="sample").sel(lag=slice(-6,6)), ls="--")
plt.plot(spags_diff.quantile(q=.9, dim="sample").sel(lag=slice(-6,6)), ls="--")
plt.plot(spags_diff.quantile(q=.1, dim="sample").sel(lag=slice(-6,6)), ls="--")
plt.axhline(0, lw=1, c="gray")
plt.show()

#### $\frac{dh}{dt}$ terms

##### early v. late

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 2.5), layout="constrained")

for (
    ax,
    v,
    n,
) in zip(axs, ["F2", "epsilon"], [T_VAR, H_VAR]):
    for comp, params_, l in zip(
        [comp_early, comp_late],
        [params_early, params_late],
        ["early", "late"],
    ):

        ax.plot(comp.lag, -comp[n] * params_[v], label=l)

    ## formatting
    f_kwargs = dict(ls="--", c="k", lw=0.8)
    ax.axhline(0, **f_kwargs)
    ax.axvline(0, **f_kwargs)
    ax.set_xlabel("Lag (months)")
    ax.axvline(6, c="gray", lw=0.5)
    ax.set_xticks([-12, 0, 6, 12])

## formattings
axs[0].set_title(r"Recharge ($-F_2 \cdot T$)")
axs[1].set_title(r"Damping ($-\varepsilon \cdot h$)")
# axs[1].set_ylabel("")/)
axs[1].legend(prop=dict(size=8))
axs[1].yaxis.set_label_position("right")
axs[1].yaxis.tick_right()
src.utils.set_lims(axs)

plt.show()

##### recharge vs damping

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 2.5), layout="constrained")

for ax, comp, params_, c in zip(
    axs,
    [comp_early, comp_late],
    [params_early, params_late],
    sns.color_palette()[:2],
):

    for v, n, ls, l in zip(
        ["F2", "epsilon"], [T_VAR, H_VAR], ["-", "--"], ["discharge", "damping"]
    ):

        ax.plot(comp.lag, -comp[n] * params_[v], ls=ls, c=c, label=l)

    ## formatting
    f_kwargs = dict(ls="--", c="k", lw=0.8)
    ax.axhline(0, **f_kwargs)
    ax.axvline(0, **f_kwargs)
    ax.set_xlabel("Lag (months)")
    ax.axvline(6, c="gray", lw=0.5)
    ax.set_xticks([-12, 0, 6, 12])

## formattings
axs[0].set_title(r"Early")
axs[1].set_title(r"Late")
axs[1].yaxis.set_label_position("right")
axs[1].yaxis.tick_right()
axs[0].legend(prop=dict(size=8))
src.utils.set_lims(axs)

plt.show()

#### $\frac{dT}{dt}$ terms

##### Early vs. Late

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 2.5), layout="constrained")

for (
    ax,
    v,
    n,
) in zip(axs, ["R", "F1"], [T_VAR, H_VAR]):
    for comp, params_, l in zip(
        [comp_early, comp_late],
        [params_early, params_late],
        ["early", "late"],
    ):

        ax.plot(comp.lag, comp[n] * params_[v], label=l)

    ## formatting
    f_kwargs = dict(ls="--", c="k", lw=0.8)
    ax.axhline(0, **f_kwargs)
    ax.axvline(0, **f_kwargs)
    ax.set_xlabel("Lag (months)")
    ax.axvline(-3, c="gray", lw=0.5)
    ax.set_xticks([-12, -3, 0, 12])

## formattings
axs[0].set_title(r"Bjerknes ($R \cdot T$)")
axs[1].set_title(r"Coupling ($F_1 \cdot h$)")
# axs[1].set_ylabel("")/)
axs[1].legend(prop=dict(size=8))
axs[1].yaxis.set_label_position("right")
axs[1].yaxis.tick_right()
src.utils.set_lims(axs)

plt.show()

##### growth vs. coupling

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 2.5), layout="constrained")

for ax, comp, params_, c in zip(
    axs,
    [comp_early, comp_late],
    [params_early, params_late],
    sns.color_palette()[:2],
):

    for v, n, ls, l in zip(
        ["R", "F1"], [T_VAR, H_VAR], ["-", "--"], ["Bjerknes", "coupling"]
    ):

        ax.plot(comp.lag, comp[n] * params_[v], ls=ls, c=c, label=l)

    ## formatting
    f_kwargs = dict(ls="--", c="k", lw=0.8)
    ax.axhline(0, **f_kwargs)
    ax.axvline(0, **f_kwargs)
    ax.set_xlabel("Lag (months)")
    ax.axvline(6, c="gray", lw=0.5)
    ax.set_xticks([-12, 0, 6, 12])

## formattings
axs[0].set_title(r"Early")
axs[1].set_title(r"Late")
axs[1].yaxis.set_label_position("right")
axs[1].yaxis.tick_right()
axs[0].legend(prop=dict(size=8))
src.utils.set_lims(axs)

plt.show()

#### Nicer spaghettie

In [ ]:
## specify variable to plot
PLOT_VAR = "T_34"
YLABEL = "K"

## plot style for spaghetti
spag_kwargs = dict(c="gray", lw=0.5, alpha=0.5)
mean_kwargs = dict(c="k", lw=2)

fig, axs = plt.subplots(1, 2, figsize=(5, 2.5), layout="constrained")

for ax, spag, ls in zip(axs, [spag_early, spag_late], ["-", "--"]):

    ## plot data
    ax.plot(comp.lag, spag[PLOT_VAR], label=l, **spag_kwargs)
    ax.plot(comp.lag, spag[PLOT_VAR].mean("sample"), ls=ls, **mean_kwargs)

    ## formatting
    f_kwargs = dict(ls="--", c="k", lw=0.8)
    ax.axhline(0, **f_kwargs)
    ax.axvline(0, **f_kwargs)
    ax.set_xticks([-12, 0, 12])
    ax.set_xlabel("Lag (months)")

## plot early composite in late
axs[1].plot(spag_early.lag, spag_early[PLOT_VAR].mean("sample"), ls="-", **mean_kwargs)

## formattings
axs[0].set_title(f"{PLOT_VAR} (Early)")
axs[1].set_title(f"{PLOT_VAR} (Late)")
axs[0].set_ylabel(YLABEL)
axs[0].set_ylabel("K")
axs[1].set_yticks([])
src.utils.set_lims(axs)

plt.show()

### spaghetti

In [ ]:
## specify variable to plot
PLOT_VAR = "dTdx"
YLABEL = "K"

## plot style for spaghetti
spag_kwargs = dict(c="gray", lw=0.5, alpha=0.5)
mean_kwargs = dict(c="k", lw=2)

fig, axs = plt.subplots(1, 2, figsize=(5, 2.5), layout="constrained")

for ax, spag, ls in zip(axs, [spag_early, spag_late], ["-", "--"]):

    ## plot data
    ax.plot(comp.lag, spag[PLOT_VAR], label=l, **spag_kwargs)
    ax.plot(comp.lag, spag[PLOT_VAR].mean("sample"), ls=ls, **mean_kwargs)

    ## formatting
    f_kwargs = dict(ls="--", c="k", lw=0.8)
    ax.axhline(0, **f_kwargs)
    ax.axvline(0, **f_kwargs)
    ax.set_xticks([-12, 0, 12])
    ax.set_xlabel("Lag (months)")

## plot early composite in late
axs[1].plot(spag_early.lag, spag_early[PLOT_VAR].mean("sample"), ls="-", **mean_kwargs)

## formattings
axs[0].set_title(f"{PLOT_VAR} (Early)")
axs[1].set_title(f"{PLOT_VAR} (Late)")
axs[0].set_ylabel(YLABEL)
axs[0].set_ylabel("K")
axs[1].set_yticks([])
src.utils.set_lims(axs)

plt.show()

### Statistics

In [ ]:
def get_pdf(x, edges, lag=12):

    ## compute pdf
    pdf, _ = src.utils.get_empirical_pdf(x.sel(lag=lag), edges=edges)

    ## get center points
    bin_centers = 0.5 * (edges[1:] + edges[:-1])

    ## return in DataArray format
    return xr.DataArray(pdf, coords=dict(centers=bin_centers))


def plot_pdf_on_ax(ax, name, pdf0, pdf1, edges):
    """plot PDFs for given name on given ax object"""

    ## plot data
    ax.stairs(pdf0[name], EDGES, lw=1.5, label="early")
    ax.stairs(pdf1[name], EDGES, fill=True, alpha=0.3, label="late")

    ## format
    ax.axvline(0, ls="-", c="k")
    ax.set_title(name)
    ax.set_yticks([])
    ax.set_xticks([-2, 0, 2])
    ax.set_ylim([0, 0.9])

    return

#### Compute PDFs

In [ ]:
## specify lag
LAG = 12

## specify edges for pdf
EDGES = np.arange(-3.75, 4.25, 0.5)
# EDGES = np.arange(-4.75, 5.25, 0.5)

## compute
pdf_early_post = spag_early.apply(get_pdf, edges=EDGES, lag=LAG)
pdf_late_post = spag_late.apply(get_pdf, edges=EDGES, lag=LAG)

pdf_early_pre = spag_early.apply(get_pdf, edges=EDGES, lag=-LAG)
pdf_late_pre = spag_late.apply(get_pdf, edges=EDGES, lag=-LAG)

#### Plot

In [ ]:
## specify variables to plot
# PLOT_VARS = ["h_w", "h", "h_e"]
PLOT_VARS = ["T_4", "T_34", "T_3"]

## set up plot
fig, axs = plt.subplots(2, 3, figsize=(8, 4), layout="constrained")

pdf_kwargs_pre = dict(pdf0=pdf_early_pre, pdf1=pdf_late_pre, edges=EDGES)
pdf_kwargs_post = dict(pdf0=pdf_early_post, pdf1=pdf_late_post, edges=EDGES)

## loop thru before/after
for j, pdf_kwargs in enumerate([pdf_kwargs_pre, pdf_kwargs_post]):

    ## loop thru variables
    for i, name in enumerate(PLOT_VARS):

        ## plot data
        plot_pdf_on_ax(axs[j, i], name=name, **pdf_kwargs)

## legend / formatting
axs[0, 0].legend(prop=dict(size=8))
for ax in axs[0, :]:
    ax.set_xticks([])
for ax in axs[1, :]:
    ax.set_title(None)


plt.show()

## Nicer plots of PDFs

In [ ]:
## pick which PDFs to plot
if comp_kwargs["is_warm"]:
    pdfs = [pdf_early_pre, pdf_late_pre]
    fname = "pdf_pre-nino"
    title = "1 year before El Niño"
else:
    pdfs = [pdf_early_post, pdf_late_post]
    fname = "pdf_post-nina"
    title = "1 year after La Niña"


## func to select data
sel = lambda x: x["T_34"].values

fig, ax = plt.subplots(figsize=(3, 3), layout="constrained")

## plot PDFs
ax.stairs(sel(pdfs[0]), EDGES, fill=True, alpha=0.1, color="k", label=1870)
ax.stairs(sel(pdfs[1]), EDGES, lw=2, label=2080, color=sns.color_palette()[1])
ax.axvline(0, ls="--", c="k", lw=0.8)
ax.set_yticks([])
ax.set_ylabel(r"Density")
ax.set_xlabel(r"$^{\circ}C$")
ax.legend()
ax.set_xlim([-3, 3])
ax.set_title(title)

## save
save(fig, fname=fname)

plt.show()